In [1]:
#WOOD Red Mark Detection 

import matplotlib.pyplot as plt
import numpy as np

from skimage import io, color, img_as_float
from skimage.morphology import disk, opening, closing, remove_small_objects
from skimage.measure import label, regionprops
from scipy.ndimage import binary_fill_holes
from matplotlib.patches import Rectangle

# Load image
img = io.imread("011.png")   # change if needed

if img.ndim == 3 and img.shape[2] == 4:
    img = img[:, :, :3]

rgb = img_as_float(img)
hsv = color.rgb2hsv(img)
gray = color.rgb2gray(img)

R = rgb[:, :, 0]
G = rgb[:, :, 1]
B = rgb[:, :, 2]

hue = hsv[:, :, 0]
sat = hsv[:, :, 1]
val = hsv[:, :, 2]

# -----------------------------
# RED PAINT MASK
# -----------------------------

red_mask = (
    ((hue < 0.05) | (hue > 0.95)) &
    (sat > 0.16) &
    (val > 0.15) &
    (R > G * 1.03) &
    (R > B * 1.03)
)

cleaned = opening(red_mask, disk(2))
cleaned = closing(cleaned, disk(5))
cleaned = binary_fill_holes(cleaned)
cleaned = remove_small_objects(cleaned, min_size=150)

# -----------------------------
# DISPLAY MASKS
# -----------------------------

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(img)
plt.title("Original Wood Image")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(red_mask, cmap="gray")
plt.title("Initial Red Detection")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(cleaned, cmap="gray")
plt.title("Cleaned Red Mask")
plt.axis("off")

plt.tight_layout()
plt.show()

# -----------------------------
# DRAW BOXES, EXCLUDING KNOT
# -----------------------------

labeled_mask = label(cleaned)
regions = regionprops(labeled_mask)

fig, ax = plt.subplots(figsize=(8,8))
ax.imshow(img)
ax.set_title("Detected Red Paint Defects")
ax.axis("off")

defect_count = 0

for region in regions:
    minr, minc, maxr, maxc = region.bbox

    region_pixels = labeled_mask[minr:maxr, minc:maxc] == region.label

    region_gray = gray[minr:maxr, minc:maxc]
    region_R = R[minr:maxr, minc:maxc]
    region_G = G[minr:maxr, minc:maxc]
    region_B = B[minr:maxr, minc:maxc]

    mean_gray = np.mean(region_gray[region_pixels])
    dark_fraction = np.mean(region_gray[region_pixels] < 0.28)

    red_strength = region_R - ((region_G + region_B) / 2)
    mean_red_strength = np.mean(red_strength[region_pixels])

    box_h = maxr - minr
    box_w = maxc - minc

    # Important:
    # The knot has a high dark_fraction, so this rejects it.
    if (
        region.area >= 150 and
        box_h > 25 and
        box_w > 25 and
        mean_red_strength > 0.025 and
        mean_gray > 0.33 and
        dark_fraction < 0.18
    ):
        rect = Rectangle(
            (minc, minr),
            box_w,
            box_h,
            linewidth=3,
            edgecolor="red",
            facecolor="none"
        )

        ax.add_patch(rect)

        ax.text(
            minc,
            minr - 10,
            "Red Paint Defect",
            color="red",
            fontsize=11,
            weight="bold"
        )

        defect_count += 1

plt.show()

print("Detected red paint defect regions:", defect_count)

FileNotFoundError: No such file: 'C:\Users\nt442\Documents\GitHub\ImageProcessingFinal\011.png'